# Hunyuan-7B MAP Training Notebook

This notebook is structured for Kaggle + H100.

- Start with `QUICK_RUN = True` to sanity-check the pipeline.
- Switch to `QUICK_RUN = False` for full training.
- The trained model is saved under the model's own name.
- Validation MAP@3 is printed after training.


In [ ]:
!pip install -q -U "transformers>=4.55.0" peft accelerate datasets scikit-learn joblib


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import inspect
import json
import random
from argparse import Namespace
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42

TRAIN_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"
OUTPUT_ROOT = Path("/kaggle/working/map_multi_llm_runs")

MODEL_NAME = "tencent/Hunyuan-7B-Instruct"
MODEL_SLUG = MODEL_NAME.split("/")[-1]
HF_TOKEN = os.getenv("HF_TOKEN")

QUICK_RUN = True
QUICK_RUN_ROWS = 4096
QUICK_RUN_TEST_SIZE = 0.10
QUICK_RUN_EPOCHS = 1
QUICK_RUN_MAX_STEPS = 40
FULL_TRAIN_EPOCHS = 3

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

MAX_LENGTH = 256
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 4
VAL_SPLIT = 0.10

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj"]
MODULES_TO_SAVE = ["score"]

RUN_NAME = "quick_run" if QUICK_RUN else "full_train"
RUN_DIR = OUTPUT_ROOT / MODEL_SLUG / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
FINAL_MODEL_DIR = RUN_DIR / MODEL_SLUG

for path in [OUTPUT_ROOT, RUN_DIR, CHECKPOINT_DIR, LOG_DIR, FINAL_MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print({
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "model_name": MODEL_NAME,
    "quick_run": QUICK_RUN,
    "model_dtype": str(MODEL_DTYPE).replace("torch.", ""),
    "gpu_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "run_dir": str(RUN_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
})


In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_map3(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    top3 = np.argsort(-probs, axis=1)[:, :3]
    matches = top3 == labels[:, None]

    score = 0.0
    for i in range(len(labels)):
        if matches[i, 0]:
            score += 1.0
        elif matches[i, 1]:
            score += 0.5
        elif matches[i, 2]:
            score += 1.0 / 3.0

    return {"map@3": score / len(labels)}


class DataProcessor:
    def __init__(self, args):
        self.args = args
        self.le = None
        self.isPreprocess = False
        self.correct_lookup = None
        self.train_df = None
        self.test_df = None

    def load_data(self):
        self.train_df = pd.read_csv(self.args.train_path)
        self.test_df = pd.read_csv(self.args.test_path)

    def get_num_classes(self):
        if not self.isPreprocess:
            raise ValueError("Please preprocess first.")
        return int(self.train_df["label"].nunique())

    def get_label_encoder(self):
        if self.le is None:
            raise ValueError("LabelEncoder not initialized. Please run preprocess first.")
        return self.le

    @staticmethod
    def format_input(row):
        correct_text = "Yes" if row["IsCorrect"] else "No"
        return (
            f"Question: {row['QuestionText']}\n"
            f"Answer: {row['MC_Answer']}\n"
            f"Correct? {correct_text}\n"
            f"Student Explanation: {row['StudentExplanation']}\n"
        )

    def preprocess(self):
        self.load_data()
        self.train_df["Misconception"] = self.train_df["Misconception"].fillna("NA")
        self.train_df["target"] = self.train_df["Category"] + ":" + self.train_df["Misconception"]

        correct_samples = self.train_df[self.train_df["Category"].str.startswith("True", na=False)].copy()
        correct_samples["count"] = correct_samples.groupby(["QuestionId", "MC_Answer"])["MC_Answer"].transform("count")
        most_popular_correct = correct_samples.sort_values("count", ascending=False).drop_duplicates(["QuestionId"])
        self.correct_lookup = most_popular_correct[["QuestionId", "MC_Answer"]].copy()
        self.correct_lookup["IsCorrect_flag"] = True

        self.train_df = self.train_df.merge(self.correct_lookup, on=["QuestionId", "MC_Answer"], how="left")
        self.train_df["IsCorrect"] = self.train_df["IsCorrect_flag"].notna()
        self.train_df = self.train_df.drop(columns=["IsCorrect_flag"])

        self.le = LabelEncoder()
        self.train_df["label"] = self.le.fit_transform(self.train_df["target"]).astype(np.int64)
        self.train_df["text"] = self.train_df.apply(self.format_input, axis=1)

        self.isPreprocess = True
        return self.train_df

    def inference_processor(self):
        if not self.isPreprocess:
            raise ValueError("Please preprocess the training data first.")
        self.test_df = self.test_df.merge(self.correct_lookup, on=["QuestionId", "MC_Answer"], how="left")
        self.test_df["IsCorrect"] = self.test_df["IsCorrect_flag"].notna()
        self.test_df = self.test_df.drop(columns=["IsCorrect_flag"])
        self.test_df["text"] = self.test_df.apply(self.format_input, axis=1)
        return self.test_df


def dataframe_to_dataset(frame: pd.DataFrame, tokenizer, max_length: int) -> Dataset:
    clean = frame[["text", "label"]].copy()
    clean["label"] = clean["label"].astype(np.int64)
    ds = Dataset.from_pandas(clean, preserve_index=False)
    ds = ds.rename_column("label", "labels")

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    return ds.map(tokenize_fn, batched=True, remove_columns=["text"])


seed_everything(SEED)


In [ ]:
args = Namespace(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    output_dir=str(FINAL_MODEL_DIR),
    mode="train",
    model_name=MODEL_NAME,
    quick_run=QUICK_RUN,
    num_epochs=QUICK_RUN_EPOCHS if QUICK_RUN else FULL_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    val_split=VAL_SPLIT,
)

DP = DataProcessor(args)
train_df = DP.preprocess()
test_df = DP.inference_processor()

le = DP.get_label_encoder()
n_classes = DP.get_num_classes()
id2label = {i: label for i, label in enumerate(le.classes_)}
label2id = {label: i for i, label in id2label.items()}

work_df = train_df.copy()
if QUICK_RUN:
    sample_size = min(QUICK_RUN_ROWS, len(work_df))
    work_df = work_df.sample(n=sample_size, random_state=SEED).reset_index(drop=True)

split_size = QUICK_RUN_TEST_SIZE if QUICK_RUN else VAL_SPLIT
stratify_labels = work_df["label"] if work_df["label"].value_counts().min() >= 2 else None
train_split_df, eval_df = train_test_split(
    work_df,
    test_size=split_size,
    random_state=SEED,
    stratify=stratify_labels,
)

train_split_df = train_split_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Working train shape: {train_split_df.shape}")
print(f"Validation shape: {eval_df.shape}")
print(f"Target classes: {n_classes}")
print("\nExample formatted prompt:\n")
print(train_split_df.loc[0, "text"])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_ds = dataframe_to_dataset(train_split_df, tokenizer, MAX_LENGTH)
eval_ds = dataframe_to_dataset(eval_df, tokenizer, MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(train_ds)
print(eval_ds)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is intended for a GPU runtime. Kaggle H100 is recommended.")

print(f"Using GPU: {torch.cuda.get_device_name(0)}")
print(f"Visible CUDA devices: {torch.cuda.device_count()}")

model_load_kwargs = {
    "pretrained_model_name_or_path": MODEL_NAME,
    "num_labels": n_classes,
    "torch_dtype": MODEL_DTYPE,
    "token": HF_TOKEN,
    "trust_remote_code": True,
    "device_map": "auto",
}

model = AutoModelForSequenceClassification.from_pretrained(**model_load_kwargs)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.id2label = id2label
model.config.label2id = label2id
model.config.problem_type = "single_label_classification"

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    modules_to_save=MODULES_TO_SAVE,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

epochs = QUICK_RUN_EPOCHS if QUICK_RUN else FULL_TRAIN_EPOCHS
max_steps = QUICK_RUN_MAX_STEPS if QUICK_RUN else -1

training_arg_values = {
    "output_dir": str(CHECKPOINT_DIR),
    "overwrite_output_dir": True,
    "do_train": True,
    "do_eval": True,
    "evaluation_strategy": "steps",
    "eval_steps": 20 if QUICK_RUN else 200,
    "save_strategy": "steps",
    "save_steps": 20 if QUICK_RUN else 200,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "map@3",
    "greater_is_better": True,
    "logging_strategy": "steps",
    "logging_steps": 10 if QUICK_RUN else 50,
    "num_train_epochs": epochs,
    "max_steps": max_steps,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "lr_scheduler_type": "cosine",
    "remove_unused_columns": False,
    "bf16": USE_BF16,
    "fp16": torch.cuda.is_available() and not USE_BF16,
    "report_to": "none",
    "disable_tqdm": True,
    "gradient_checkpointing": True,
    "dataloader_drop_last": False,
    "dataloader_pin_memory": False,
    "optim": "adamw_torch",
}

training_arg_signature = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_arg_signature and "evaluation_strategy" not in training_arg_signature:
    training_arg_values["eval_strategy"] = training_arg_values.pop("evaluation_strategy")

supported_training_args = {
    key: value
    for key, value in training_arg_values.items()
    if key in training_arg_signature and value is not None
}

training_args = TrainingArguments(**supported_training_args)

trainer_signature = inspect.signature(Trainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": eval_ds,
    "compute_metrics": compute_map3,
    "data_collator": data_collator,
}

if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

supported_trainer_kwargs = {
    key: value
    for key, value in trainer_kwargs.items()
    if key in trainer_signature and value is not None
}

trainer = Trainer(**supported_trainer_kwargs)

for callback in list(trainer.callback_handler.callbacks):
    if callback.__class__.__name__ == "NotebookProgressCallback":
        trainer.remove_callback(callback.__class__)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()
metrics = dict(train_result.metrics)
metrics.update(trainer.evaluate())

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
joblib.dump(le, RUN_DIR / "label_encoder.joblib")

summary = {
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,
    "quick_run": QUICK_RUN,
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "run_dir": str(RUN_DIR),
    "final_model_dir": str(FINAL_MODEL_DIR),
    "n_classes": n_classes,
    "train_rows": int(len(train_split_df)),
    "eval_rows": int(len(eval_df)),
    "metrics": metrics,
}

with open(RUN_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Training complete.")
print(f"Saved model to: {FINAL_MODEL_DIR}")
print(f"Saved label encoder to: {RUN_DIR / 'label_encoder.joblib'}")
print(f"Validation MAP@3: {metrics.get('eval_map@3')}")
print(json.dumps(metrics, indent=2))


In [ ]:
sample_text = test_df.loc[0, "text"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
target_device = next(model.parameters()).device
inputs = {key: value.to(target_device) for key, value in inputs.items()}

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits

top_k = min(3, logits.shape[-1])
top3 = torch.topk(logits, k=top_k, dim=-1).indices[0].detach().cpu().numpy()
print("Sample top predictions:")
print(le.inverse_transform(top3))
